# Erasus — Coreset Ablation Benchmark (GPU)
Running coreset fraction ablation on a trained model to validate the core thesis: small coresets approximate full-set unlearning.

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git@docs/rewrite-readme matplotlib

In [ ]:
"""
Coreset Ablation on a TRAINED model with real memorization.
We train a CNN on CIFAR-10 subset, then measure unlearning quality
at different coreset fractions to validate that 10% coreset ≈ 100%.
"""
import copy
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes)
        )

    def forward(self, x, labels=None, **kwargs):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        logits = self.classifier(x)
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            return type("Out", (), {"logits": logits, "loss": loss})()
        return logits

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
full_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

np.random.seed(42)
forget_indices = [i for i, (_, y) in enumerate(full_train) if y in [0, 1]][:500]
retain_indices = [i for i, (_, y) in enumerate(full_train) if y not in [0, 1]][:1500]

forget_set = Subset(full_train, forget_indices)
retain_set = Subset(full_train, retain_indices)

forget_loader = DataLoader(forget_set, batch_size=64, shuffle=False)
retain_loader = DataLoader(retain_set, batch_size=64, shuffle=False)

all_indices = forget_indices + retain_indices
train_set = Subset(full_train, all_indices)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

print(f"Forget set: {len(forget_set)} samples (classes 0,1)")
print(f"Retain set: {len(retain_set)} samples (classes 2-9)")
print(f"Total training: {len(train_set)} samples")

In [ ]:
model = SmallCNN(n_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training model to memorize the data...")
for epoch in range(20):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = F.cross_entropy(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/20  Loss: {total_loss / len(train_loader):.4f}")

@torch.no_grad()
def compute_accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        logits = out.logits if hasattr(out, 'logits') else out
        preds = logits.argmax(dim=-1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / max(total, 1)

base_forget_acc = compute_accuracy(model, forget_loader)
base_retain_acc = compute_accuracy(model, retain_loader)
print(f"\nTrained model \u2014 Forget Acc: {base_forget_acc:.4f}, Retain Acc: {base_retain_acc:.4f}")

trained_state = copy.deepcopy(model.state_dict())

In [ ]:
import erasus.strategies
import erasus.selectors
from erasus.unlearners import ErasusUnlearner

FRACTIONS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
SELECTORS = ["influence", "gradient_norm", "el2n", "tracin", "herding", "kcenter"]
STRATEGY = "gradient_ascent"
EPOCHS = 5

all_results = {}

for sel in SELECTORS:
    print(f"\n{'='*60}")
    print(f"  Selector: {sel}")
    print(f"{'='*60}")
    sel_results = []
    for frac in FRACTIONS:
        test_model = SmallCNN(n_classes=10).to(device)
        test_model.load_state_dict(copy.deepcopy(trained_state))
        try:
            unlearner = ErasusUnlearner(
                model=test_model, strategy=STRATEGY,
                selector=sel if frac < 1.0 else None,
                device=str(device), strategy_kwargs={"lr": 1e-3},
            )
            t0 = time.perf_counter()
            result = unlearner.fit(
                forget_data=forget_loader, retain_data=retain_loader,
                prune_ratio=frac if frac < 1.0 else None, epochs=EPOCHS,
            )
            elapsed = time.perf_counter() - t0
            forget_acc = compute_accuracy(unlearner.model, forget_loader)
            retain_acc = compute_accuracy(unlearner.model, retain_loader)
            sel_results.append({"fraction": frac, "forget_accuracy": round(forget_acc, 4),
                "retain_accuracy": round(retain_acc, 4), "time_s": round(elapsed, 3), "status": "OK"})
            print(f"  {frac*100:5.1f}% | F.Acc: {forget_acc:.4f} | R.Acc: {retain_acc:.4f} | {elapsed:.2f}s")
        except Exception as e:
            sel_results.append({"fraction": frac, "forget_accuracy": None, "retain_accuracy": None,
                "time_s": 0, "status": "ERROR", "error": str(e)[:100]})
            print(f"  {frac*100:5.1f}% | ERROR: {str(e)[:80]}")
    all_results[sel] = sel_results

print(f"\n{'='*60}")
print(f"  Selector: random (baseline)")
print(f"{'='*60}")
random_results = []
for frac in FRACTIONS:
    test_model = SmallCNN(n_classes=10).to(device)
    test_model.load_state_dict(copy.deepcopy(trained_state))
    try:
        unlearner = ErasusUnlearner(
            model=test_model, strategy=STRATEGY,
            selector="random" if frac < 1.0 else None,
            device=str(device), strategy_kwargs={"lr": 1e-3},
        )
        t0 = time.perf_counter()
        result = unlearner.fit(forget_data=forget_loader, retain_data=retain_loader,
                               prune_ratio=frac if frac < 1.0 else None, epochs=EPOCHS)
        elapsed = time.perf_counter() - t0
        forget_acc = compute_accuracy(unlearner.model, forget_loader)
        retain_acc = compute_accuracy(unlearner.model, retain_loader)
        random_results.append({"fraction": frac, "forget_accuracy": round(forget_acc, 4),
                               "retain_accuracy": round(retain_acc, 4), "time_s": round(elapsed, 3), "status": "OK"})
        print(f"  {frac*100:5.1f}% | F.Acc: {forget_acc:.4f} | R.Acc: {retain_acc:.4f} | {elapsed:.2f}s")
    except Exception as e:
        random_results.append({"fraction": frac, "forget_accuracy": None, "retain_accuracy": None,
                               "time_s": 0, "status": "ERROR", "error": str(e)[:100]})
        print(f"  {frac*100:5.1f}% | ERROR: {str(e)[:80]}")

print("\nAblation complete!")

In [ ]:
colors = {
    'influence': '#e41a1c', 'gradient_norm': '#377eb8', 'el2n': '#4daf4a',
    'tracin': '#984ea3', 'herding': '#ff7f00', 'kcenter': '#a65628',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
ax1, ax2, ax3 = axes

for sel, points in all_results.items():
    ok = [p for p in points if p['status'] == 'OK']
    fracs = [p['fraction'] for p in ok]
    f_accs = [p['forget_accuracy'] for p in ok]
    r_accs = [p['retain_accuracy'] for p in ok]
    scores = [(1 - p['forget_accuracy']) * p['retain_accuracy'] for p in ok]
    c = colors.get(sel, '#999')
    ax1.plot(fracs, f_accs, 'o-', color=c, label=sel, markersize=5, linewidth=1.5)
    ax2.plot(fracs, r_accs, 'o-', color=c, label=sel, markersize=5, linewidth=1.5)
    ax3.plot(fracs, scores, 'o-', color=c, label=sel, markersize=5, linewidth=1.5)

ok_rand = [p for p in random_results if p['status'] == 'OK']
ax1.plot([p['fraction'] for p in ok_rand], [p['forget_accuracy'] for p in ok_rand], 'x--', color='#999', label='random', markersize=6)
ax2.plot([p['fraction'] for p in ok_rand], [p['retain_accuracy'] for p in ok_rand], 'x--', color='#999', label='random', markersize=6)
ax3.plot([p['fraction'] for p in ok_rand], [(1-p['forget_accuracy'])*p['retain_accuracy'] for p in ok_rand], 'x--', color='#999', label='random', markersize=6)

ax1.axhline(y=base_forget_acc, color='black', linestyle=':', alpha=0.5, label=f'base ({base_forget_acc:.2f})')
ax2.axhline(y=base_retain_acc, color='black', linestyle=':', alpha=0.5, label=f'base ({base_retain_acc:.2f})')

for ax in axes:
    ax.set_xlabel('Coreset Fraction', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
ax1.set_ylabel('Forget Accuracy (lower = better)')
ax1.set_title('Forgetting Quality', fontweight='bold')
ax2.set_ylabel('Retain Accuracy (higher = better)')
ax2.set_title('Utility Preservation', fontweight='bold')
ax3.set_ylabel('Tradeoff Score (higher = better)')
ax3.set_title('Combined Score', fontweight='bold')
fig.suptitle('Coreset Unlearning on CIFAR-10 CNN (Tesla T4 GPU)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

print("\n" + "="*80)
print(f"  BASE MODEL: Forget Acc = {base_forget_acc:.4f}, Retain Acc = {base_retain_acc:.4f}")
print("="*80)
print(f"  {'Selector':<16} {'10% F.Acc':>10} {'10% R.Acc':>10} {'100% F.Acc':>10} {'100% R.Acc':>10} {'R.Acc Gap':>10}")
print("-"*80)
for sel, points in {**all_results, 'random': random_results}.items():
    ok = [p for p in points if p['status'] == 'OK']
    p10 = next((p for p in ok if abs(p['fraction'] - 0.1) < 0.05), None)
    p100 = next((p for p in ok if p['fraction'] >= 1.0), None)
    if p10 and p100:
        gap = p10['retain_accuracy'] - p100['retain_accuracy']
        print(f"  {sel:<16} {p10['forget_accuracy']:>10.4f} {p10['retain_accuracy']:>10.4f} "
              f"{p100['forget_accuracy']:>10.4f} {p100['retain_accuracy']:>10.4f} {gap:>+10.4f}")
print("="*80)